# P9-IMPL-USGS-SEISMIC-INGEST — USGS Earthquake Ingestion Pipeline

**Author:** @data-engineer | **Date:** 2026-10-22  
**Schema:** `cdm_tmforum.provable_resilience_intelligence`  
**Source:** [earthquake.usgs.gov/fdsnws/event/1](https://earthquake.usgs.gov/fdsnws/event/1)  
**Refresh:** 30-minute cadence (scheduled)  
**SID Domain:** Common → GeographicPlace → HazardEvent (seismic extension)  

## Pipeline Architecture
```
USGS FDSNWS API (GeoJSON) → Fetch+Parse → GeoJSON→WKT POINT → stg_usgs_earthquake (Bronze)
                                                              ↓ MERGE
                                                         geographic_hazard_event (Gold)
```

## Key Transforms
- UNIX epoch ms → TIMESTAMP
- estimated_radius_km = 10^(0.5 * magnitude - 1.0)
- GeoJSON Point → WKT POINT
- PAGER alert_level → PRI severity mapping

## Lakelink Fiber — Provable Resilience Intelligence

In [0]:
%sql
-- Create the stg_usgs_earthquake staging table (idempotent)
CREATE TABLE IF NOT EXISTS cdm_tmforum.provable_resilience_intelligence.stg_usgs_earthquake (
  event_id              STRING        NOT NULL  COMMENT 'USGS event identifier (e.g. us7000abcd). Globally unique.',
  magnitude             DOUBLE        NOT NULL  COMMENT 'Earthquake magnitude on the scale indicated by mag_type',
  mag_type              STRING        NOT NULL  COMMENT 'Magnitude type: ml (local), mb (body), mw (moment), ms (surface)',
  event_type            STRING        NOT NULL  COMMENT 'Event type: earthquake | quarry blast | explosion | ice quake',
  alert_level           STRING                  COMMENT 'PAGER alert level: green | yellow | orange | red. NULL if not computed.',
  significance          INT                     COMMENT 'USGS significance score (0-1000). Higher = more significant.',
  latitude              DOUBLE        NOT NULL  COMMENT 'WGS84 latitude of earthquake epicenter',
  longitude             DOUBLE        NOT NULL  COMMENT 'WGS84 longitude of earthquake epicenter',
  depth_km              DOUBLE        NOT NULL  COMMENT 'Depth of earthquake hypocenter in kilometers',
  place_description     STRING                  COMMENT 'Human-readable location (e.g. 15 km NNE of Oklahoma City)',
  geom_wkt              STRING        NOT NULL  COMMENT 'WKT POINT geometry from GeoJSON coordinates. WGS84.',
  tsunami_flag          INT                     COMMENT 'Tsunami advisory: 0 = no, 1 = yes',
  felt_reports          INT                     COMMENT 'Number of DYFI (Did You Feel It?) reports received',
  cdi                   DOUBLE                  COMMENT 'Community Determined Intensity (Mercalli scale, 1-10)',
  mmi                   DOUBLE                  COMMENT 'Maximum Modified Mercalli Intensity (ShakeMap)',
  event_time            TIMESTAMP     NOT NULL  COMMENT 'Origin time of the earthquake (UTC, from UNIX epoch ms)',
  updated_time          TIMESTAMP               COMMENT 'Last update time from USGS (UTC)',
  review_status         STRING                  COMMENT 'Review status: automatic | reviewed | deleted',
  estimated_radius_km   DOUBLE                  COMMENT 'Estimated impact radius based on magnitude: 10^(0.5*mag-1.0) km. For buffer zone spatial join.',
  source_url            STRING        NOT NULL  COMMENT 'USGS detail GeoJSON URL for this event',
  ingested_at           TIMESTAMP     NOT NULL  COMMENT 'UTC timestamp when row was ingested into staging',
  raw_json              STRING                  COMMENT 'Original GeoJSON feature (for audit/replay)'
)
COMMENT 'Staging: USGS earthquake events from FDSNWS API. One row per seismic event. Refreshed every 30 min. Source for geographic_hazard_event earthquake enrichment. Lakelink Fiber PRI.'
TBLPROPERTIES (
  'quality.pipeline' = 'bronze',
  'refresh.cadence' = 'PT30M',
  'source.api' = 'earthquake.usgs.gov/fdsnws/event/1',
  'sid.concept' = 'Common.GeographicPlace.HazardEvent'
)

In [0]:
"""Fetch USGS earthquake events (M2.5+ in past 30 days) and transform GeoJSON→WKT for Lakelink Fiber PRI."""
import requests
import json
import math
from datetime import datetime, timezone, timedelta
from pyspark.sql import Row
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, IntegerType, TimestampType
)

# --- API Configuration ---
USGS_API_URL = "https://earthquake.usgs.gov/fdsnws/event/1/query"
SCHEMA = "cdm_tmforum.provable_resilience_intelligence"

# Query: M2.5+ earthquakes in the past 30 days, CONUS bounding box
params = {
    "format": "geojson",
    "starttime": (datetime.now(timezone.utc) - timedelta(days=30)).strftime("%Y-%m-%d"),
    "endtime": datetime.now(timezone.utc).strftime("%Y-%m-%d"),
    "minmagnitude": 2.5,
    "minlatitude": 24.0,
    "maxlatitude": 50.0,
    "minlongitude": -125.0,
    "maxlongitude": -65.0,
    "orderby": "time"
}

print(f"Fetching USGS earthquakes: M2.5+, CONUS, past 30 days...")
resp = requests.get(USGS_API_URL, params=params, timeout=60)
resp.raise_for_status()
data = resp.json()
features = data.get("features", [])
print(f"Fetched {len(features)} earthquake events from USGS FDSNWS")

# --- Transform GeoJSON features to rows ---
def epoch_ms_to_ts(epoch_ms):
    """Convert UNIX epoch milliseconds to datetime."""
    if epoch_ms is None:
        return None
    return datetime.fromtimestamp(epoch_ms / 1000.0, tz=timezone.utc)

def compute_estimated_radius(mag):
    """Estimated impact radius: 10^(0.5*mag - 1.0) km per P8 spec."""
    if mag is None:
        return None
    return round(math.pow(10, 0.5 * mag - 1.0), 2)

now_utc = datetime.now(timezone.utc)
rows = []
skipped = 0

for f in features:
    props = f.get("properties", {})
    geom = f.get("geometry", {})
    coords = geom.get("coordinates", [None, None, None])  # [lon, lat, depth]
    
    event_id = f.get("id")
    mag = props.get("mag")
    lon = coords[0] if len(coords) > 0 else None
    lat = coords[1] if len(coords) > 1 else None
    depth = coords[2] if len(coords) > 2 else None
    
    # Skip events missing required fields
    if not all([event_id, mag is not None, lat is not None, lon is not None, depth is not None]):
        skipped += 1
        continue
    
    geom_wkt = f"POINT({lon} {lat})"
    
    rows.append(Row(
        event_id=str(event_id),
        magnitude=float(mag),
        mag_type=str(props.get("magType", "unknown")),
        event_type=str(props.get("type", "earthquake")),
        alert_level=props.get("alert"),
        significance=int(props["sig"]) if props.get("sig") is not None else None,
        latitude=float(lat),
        longitude=float(lon),
        depth_km=float(depth),
        place_description=props.get("place"),
        geom_wkt=geom_wkt,
        tsunami_flag=int(props["tsunami"]) if props.get("tsunami") is not None else None,
        felt_reports=int(props["felt"]) if props.get("felt") is not None else None,
        cdi=float(props["cdi"]) if props.get("cdi") is not None else None,
        mmi=float(props["mmi"]) if props.get("mmi") is not None else None,
        event_time=epoch_ms_to_ts(props.get("time")),
        updated_time=epoch_ms_to_ts(props.get("updated")),
        review_status=props.get("status"),
        estimated_radius_km=compute_estimated_radius(mag),
        source_url=str(props.get("detail", f"https://earthquake.usgs.gov/earthquakes/eventpage/{event_id}")),
        ingested_at=now_utc,
        raw_json=json.dumps(f)
    ))

print(f"Transformed: {len(rows)} events ready for upsert ({skipped} skipped for missing fields)")
if rows:
    print(f"Magnitude range: {min(r.magnitude for r in rows):.1f} - {max(r.magnitude for r in rows):.1f}")
    print(f"Radius range: {min(r.estimated_radius_km for r in rows):.1f} - {max(r.estimated_radius_km for r in rows):.1f} km")

In [0]:
# Upsert transformed earthquakes into staging table
if len(rows) == 0:
    print("⚠️ No earthquakes to upsert. USGS may have returned empty results.")
else:
    schema = StructType([
        StructField("event_id", StringType(), False),
        StructField("magnitude", DoubleType(), False),
        StructField("mag_type", StringType(), False),
        StructField("event_type", StringType(), False),
        StructField("alert_level", StringType(), True),
        StructField("significance", IntegerType(), True),
        StructField("latitude", DoubleType(), False),
        StructField("longitude", DoubleType(), False),
        StructField("depth_km", DoubleType(), False),
        StructField("place_description", StringType(), True),
        StructField("geom_wkt", StringType(), False),
        StructField("tsunami_flag", IntegerType(), True),
        StructField("felt_reports", IntegerType(), True),
        StructField("cdi", DoubleType(), True),
        StructField("mmi", DoubleType(), True),
        StructField("event_time", TimestampType(), False),
        StructField("updated_time", TimestampType(), True),
        StructField("review_status", StringType(), True),
        StructField("estimated_radius_km", DoubleType(), True),
        StructField("source_url", StringType(), False),
        StructField("ingested_at", TimestampType(), False),
        StructField("raw_json", StringType(), True)
    ])
    
    df = spark.createDataFrame(rows, schema=schema)
    df.createOrReplaceTempView("stg_batch")
    
    result = spark.sql(f"""
        MERGE INTO {SCHEMA}.stg_usgs_earthquake AS target
        USING stg_batch AS source
        ON target.event_id = source.event_id
        WHEN MATCHED AND source.updated_time > target.updated_time THEN UPDATE SET
            target.magnitude = source.magnitude,
            target.mag_type = source.mag_type,
            target.event_type = source.event_type,
            target.alert_level = source.alert_level,
            target.significance = source.significance,
            target.latitude = source.latitude,
            target.longitude = source.longitude,
            target.depth_km = source.depth_km,
            target.place_description = source.place_description,
            target.geom_wkt = source.geom_wkt,
            target.tsunami_flag = source.tsunami_flag,
            target.felt_reports = source.felt_reports,
            target.cdi = source.cdi,
            target.mmi = source.mmi,
            target.event_time = source.event_time,
            target.updated_time = source.updated_time,
            target.review_status = source.review_status,
            target.estimated_radius_km = source.estimated_radius_km,
            target.source_url = source.source_url,
            target.ingested_at = source.ingested_at,
            target.raw_json = source.raw_json
        WHEN NOT MATCHED THEN INSERT *
    """)
    
    metrics = result.first()
    print(f"✅ MERGE complete: {metrics['num_inserted_rows']} inserted, {metrics['num_updated_rows']} updated")
    
    # Verify staging table
    total = spark.sql(f"SELECT COUNT(*) FROM {SCHEMA}.stg_usgs_earthquake").first()[0]
    print(f"Staging total: {total} earthquake events")

In [0]:
# Enrich geographic_hazard_event with new earthquake events
# Map PAGER alert_level to PRI severity; compute estimated_radius_km

result = spark.sql(f"""
    MERGE INTO {SCHEMA}.geographic_hazard_event AS gold
    USING (
        SELECT
            CONCAT('HAZ-USGS-', SUBSTR(MD5(event_id), 1, 16)) AS hazard_id,
            'earthquake' AS hazard_type,
            CASE alert_level
                WHEN 'red' THEN 'critical'
                WHEN 'orange' THEN 'high'
                WHEN 'yellow' THEN 'medium'
                WHEN 'green' THEN 'low'
                ELSE CASE
                    WHEN magnitude >= 6.0 THEN 'critical'
                    WHEN magnitude >= 5.0 THEN 'high'
                    WHEN magnitude >= 4.0 THEN 'medium'
                    ELSE 'low'
                END
            END AS severity,
            latitude,
            longitude,
            estimated_radius_km AS radius_km,
            geom_wkt,
            event_time AS event_start,
            CAST(NULL AS TIMESTAMP) AS event_end,
            'usgs' AS source,
            CONCAT('M', CAST(magnitude AS STRING), ' ', COALESCE(place_description, 'Unknown location'),
                   ' (depth: ', CAST(ROUND(depth_km, 1) AS STRING), ' km)') AS description,
            CURRENT_TIMESTAMP() AS created_at
        FROM {SCHEMA}.stg_usgs_earthquake
    ) AS stg
    ON gold.hazard_id = stg.hazard_id
    WHEN MATCHED THEN UPDATE SET
        gold.severity = stg.severity,
        gold.latitude = stg.latitude,
        gold.longitude = stg.longitude,
        gold.radius_km = stg.radius_km,
        gold.geom_wkt = stg.geom_wkt,
        gold.event_start = stg.event_start,
        gold.source = stg.source,
        gold.description = stg.description
    WHEN NOT MATCHED THEN INSERT *
""")

metrics = result.first()
print(f"✅ Gold MERGE complete: {metrics['num_inserted_rows']} inserted, {metrics['num_updated_rows']} updated")

# Verify gold enrichment
gold_usgs = spark.sql(f"SELECT COUNT(*) FROM {SCHEMA}.geographic_hazard_event WHERE source = 'usgs' AND hazard_type = 'earthquake'").first()[0]
print(f"Gold earthquake events total: {gold_usgs}")

In [0]:
# Spatial intersection: find spans within earthquake estimated_radius_km
# Uses Haversine distance on epicenter vs span midpoint_lat/midpoint_lon
# Follows the established hazard_span_intersection_pipeline pattern:
#   - intersection_id = SHA2(source || source_id || span_id, 256)
#   - proximity_band: direct_impact (<1km) | near_field (1-5km) | buffer_zone (5-radius)
# NOTE: Real USGS earthquakes (M2.5-4.2, radius 1.8-11.9km) may produce 0 intersections
# if no Lakelink Fiber spans are within the small impact radii. This is expected.

from datetime import datetime, timezone
batch_id = f"batch-{datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')}"

result = spark.sql(f"""
    MERGE INTO {SCHEMA}.hazard_span_intersection AS hsi
    USING (
        SELECT
            SHA2(CONCAT('usgs', eq.event_id, s.span_id), 256) AS intersection_id,
            'usgs' AS hazard_source,
            eq.event_id AS hazard_source_id,
            'earthquake' AS hazard_type,
            CASE
                WHEN eq.magnitude >= 6.0 THEN 'critical'
                WHEN eq.magnitude >= 5.0 THEN 'high'
                WHEN eq.magnitude >= 4.0 THEN 'medium'
                ELSE 'low'
            END AS hazard_severity,
            eq.latitude AS hazard_lat,
            eq.longitude AS hazard_lon,
            eq.estimated_radius_km AS hazard_radius_km,
            eq.event_time AS hazard_start,
            CAST(NULL AS TIMESTAMP) AS hazard_end,
            s.span_id,
            s.midpoint_lat AS span_midpoint_lat,
            s.midpoint_lon AS span_midpoint_lon,
            -- Haversine distance in km
            2 * 6371 * ASIN(SQRT(
                POWER(SIN(RADIANS(s.midpoint_lat - eq.latitude) / 2), 2) +
                COS(RADIANS(eq.latitude)) * COS(RADIANS(s.midpoint_lat)) *
                POWER(SIN(RADIANS(s.midpoint_lon - eq.longitude) / 2), 2)
            )) AS distance_km,
            CASE
                WHEN 2 * 6371 * ASIN(SQRT(
                    POWER(SIN(RADIANS(s.midpoint_lat - eq.latitude) / 2), 2) +
                    COS(RADIANS(eq.latitude)) * COS(RADIANS(s.midpoint_lat)) *
                    POWER(SIN(RADIANS(s.midpoint_lon - eq.longitude) / 2), 2)
                )) < 1.0 THEN 'direct_impact'
                WHEN 2 * 6371 * ASIN(SQRT(
                    POWER(SIN(RADIANS(s.midpoint_lat - eq.latitude) / 2), 2) +
                    COS(RADIANS(eq.latitude)) * COS(RADIANS(s.midpoint_lat)) *
                    POWER(SIN(RADIANS(s.midpoint_lon - eq.longitude) / 2), 2)
                )) < 5.0 THEN 'near_field'
                ELSE 'buffer_zone'
            END AS proximity_band,
            CONCAT('HAZ-USGS-', SUBSTR(MD5(eq.event_id), 1, 16)) AS hazard_id,
            '{batch_id}' AS batch_id,
            CURRENT_TIMESTAMP() AS computed_at
        FROM {SCHEMA}.stg_usgs_earthquake eq
        CROSS JOIN {SCHEMA}.physical_resource_span s
        WHERE
            ABS(s.midpoint_lat - eq.latitude) < (eq.estimated_radius_km / 111.0) * 1.5
            AND ABS(s.midpoint_lon - eq.longitude) < (eq.estimated_radius_km / (111.0 * COS(RADIANS(eq.latitude)))) * 1.5
            AND 2 * 6371 * ASIN(SQRT(
                POWER(SIN(RADIANS(s.midpoint_lat - eq.latitude) / 2), 2) +
                COS(RADIANS(eq.latitude)) * COS(RADIANS(s.midpoint_lat)) *
                POWER(SIN(RADIANS(s.midpoint_lon - eq.longitude) / 2), 2)
            )) <= eq.estimated_radius_km
    ) AS new_ix
    ON hsi.intersection_id = new_ix.intersection_id
    WHEN NOT MATCHED THEN INSERT (
        intersection_id, hazard_source, hazard_source_id, hazard_type, hazard_severity,
        hazard_lat, hazard_lon, hazard_radius_km, hazard_start, hazard_end,
        span_id, span_midpoint_lat, span_midpoint_lon, distance_km, is_within_radius,
        proximity_band, computed_at, batch_id, hazard_id
    ) VALUES (
        new_ix.intersection_id, new_ix.hazard_source, new_ix.hazard_source_id, new_ix.hazard_type, new_ix.hazard_severity,
        new_ix.hazard_lat, new_ix.hazard_lon, new_ix.hazard_radius_km, new_ix.hazard_start, new_ix.hazard_end,
        new_ix.span_id, new_ix.span_midpoint_lat, new_ix.span_midpoint_lon, new_ix.distance_km, TRUE,
        new_ix.proximity_band, new_ix.computed_at, new_ix.batch_id, new_ix.hazard_id
    )
""")

metrics = result.first()
print(f"✅ Hazard-span intersection: {metrics['num_inserted_rows']} new from live USGS data")
if metrics['num_inserted_rows'] == 0:
    print("   (Expected: real M2.5-4.2 earthquakes have small radii — most are in remote areas outside Lakelink Fiber network coverage)")

total_hsi = spark.sql(f"SELECT COUNT(*) FROM {SCHEMA}.hazard_span_intersection WHERE hazard_source = 'usgs'").first()[0]
print(f"Total USGS earthquake-span intersections (incl. synthetic): {total_hsi}")

In [0]:
# Final pipeline run summary
from datetime import datetime, timezone

stg_total = spark.sql(f"SELECT COUNT(*) FROM {SCHEMA}.stg_usgs_earthquake").first()[0]
gold_total = spark.sql(f"SELECT COUNT(*) FROM {SCHEMA}.geographic_hazard_event").first()[0]
gold_usgs_eq = spark.sql(f"SELECT COUNT(*) FROM {SCHEMA}.geographic_hazard_event WHERE source = 'usgs' AND hazard_type = 'earthquake'").first()[0]
hsi_usgs = spark.sql(f"SELECT COUNT(*) FROM {SCHEMA}.hazard_span_intersection WHERE hazard_source = 'usgs'").first()[0]

mag_dist = spark.sql(f"""
    SELECT 
        CASE 
            WHEN magnitude >= 6.0 THEN 'M6.0+ (critical)'
            WHEN magnitude >= 5.0 THEN 'M5.0-5.9 (high)'
            WHEN magnitude >= 4.0 THEN 'M4.0-4.9 (medium)'
            ELSE 'M2.5-3.9 (low)'
        END AS mag_bucket,
        COUNT(*) as cnt
    FROM {SCHEMA}.stg_usgs_earthquake
    GROUP BY 1 ORDER BY 1
""").collect()

print("=" * 60)
print("USGS EARTHQUAKE INGESTION — RUN COMPLETE")
print(f"Timestamp: {datetime.now(timezone.utc).isoformat()}")
print(f"Source: earthquake.usgs.gov/fdsnws/event/1")
print("=" * 60)
print(f"Staging (stg_usgs_earthquake): {stg_total} events")
print(f"Gold (geographic_hazard_event): {gold_total} total, {gold_usgs_eq} earthquake")
print(f"Hazard-span intersections (USGS): {hsi_usgs}")
print("---")
print("Magnitude distribution:")
for row in mag_dist:
    print(f"  {row['mag_bucket']}: {row['cnt']}")
print("=" * 60)